<a href="https://colab.research.google.com/github/TheLedAlchemist/ARENA_materials/blob/main/arena_cnns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone --filter=blob:none --sparse https://github.com/TheLedAlchemist/ARENA_materials.git

fatal: destination path 'ARENA_materials' already exists and is not an empty directory.


In [7]:
%cd ARENA_materials
!git sparse-checkout set chapter0_fundamentals

/content/ARENA_materials
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 69 (delta 9), reused 2 (delta 2), pack-reused 32 (from 1)
Receiving objects: 100% (69/69), 1.67 MiB | 2.30 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (69/69), done.


In [88]:
!pip install torchinfo
!pip install jaxtyping
!pip install einsum

In [90]:
import json
import sys
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path

import einops
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchinfo
from IPython.display import display
from jaxtyping import Float, Int
from PIL import Image
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from torchvision.transforms import v2 as transforms
from tqdm.auto import tqdm

# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part2_cnns"
root_dir = Path("/content/ARENA_materials")
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part2_cnns.tests as tests
import part2_cnns.utils as utils
from fundamentals_utils import CIFAR10
from plotly_utils import line

# On machines with many cores the default of one
# thread per core makes small CPU tensor operations much slower
t.set_num_threads(min(4, t.get_num_threads()))


In [29]:
print(Path.cwd())
print((Path.cwd().parents))
(Path.cwd().parents[0] / chapter)

/content/ARENA_materials
<PosixPath.parents>


PosixPath('/content/chapter0_fundamentals')

In [32]:
[p for p in Path.cwd().parents]

[PosixPath('/content'), PosixPath('/')]

## Relu

In [38]:
class MyReLU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return torch.maximum(x, torch.tensor(0.0))

In [39]:
tests.test_relu(MyReLU)

All tests in `test_relu` passed!


## Standard @ vesion

In [84]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias=True):
        """
        A simple linear (technically, affine) transformation.

        The fields should be named `weight` and `bias` for compatibility with PyTorch.
        If `bias` is False, set `self.bias` to None.
        """
        super().__init__()
        self.weight = nn.Parameter(
            torch.tensor(
                (torch.rand(out_features, in_features) - 0.5) * 2.0 / np.sqrt(in_features)
            )
        )
        self.bias = None

        if bias:
          self.bias = nn.Parameter((torch.rand(out_features) - 0.5) * 2.0 / np.sqrt(in_features))

    def __call__(self, x: Tensor) -> Tensor:
        return self.forward(x)

    def forward(self, x: Tensor) -> Tensor:
        """
        x: shape (*, in_features)
        Return: shape (*, out_features)
        """
        ret = x @ self.weight.T

        if self.bias is not None:
          ret += self.bias

        return ret

    def extra_repr(self) -> str:
        print(f"Weights shape: {self.weight.shape}\n{self.weight}\n\n")

        if self.bias is not None:
          print(f"Bias shape: {self.bias.shape}\n{self.bias}")

## Einsum version

In [91]:
torch.einsum

<function torch.functional.einsum(*args: Any) -> torch.Tensor>

In [94]:
class EinsumLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias=True):
        """
        A simple linear (technically, affine) transformation.

        The fields should be named `weight` and `bias` for compatibility with PyTorch.
        If `bias` is False, set `self.bias` to None.
        """
        super().__init__()
        self.weight = nn.Parameter(
            torch.tensor(
                (torch.rand(out_features, in_features) - 0.5) * 2.0 / np.sqrt(in_features)
            )
        )
        self.bias = None

        if bias:
          self.bias = nn.Parameter((torch.rand(out_features) - 0.5) * 2.0 / np.sqrt(in_features))

    def __call__(self, x: Tensor) -> Tensor:
        return self.forward(x)

    def forward(self, x: Tensor) -> Tensor:
        """
        x: shape (*, in_features)
        Return: shape (*, out_features)
        """
        ret = torch.einsum("...j,...ij->...i", x, self.weight)

        if self.bias is not None:
          ret += self.bias

        return ret

    def extra_repr(self) -> str:
        print(f"Weights shape: {self.weight.shape}\n{self.weight}\n\n")

        if self.bias is not None:
          print(f"Bias shape: {self.bias.shape}\n{self.bias}")

In [95]:
linear_test = Linear(3, 7, bias=True)
linear_test(torch.rand(2, 15, 3))

einsum_test = EinsumLinear(3, 7, bias=True)
einsum_test(torch.rand(2, 15, 3))

/tmp/ipykernel_526/3773404872.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


tensor([[[ 0.4129, -0.5351,  0.1255,  0.4715, -0.1104, -0.3136, -1.0771],
         [ 0.4142, -0.5107,  0.1428,  0.4652, -0.1273, -0.3025, -1.0108],
         [ 0.5305, -0.3174, -0.0431,  0.1034, -0.1403, -0.0099, -0.6913],
         [ 0.4012, -0.4376,  0.1949,  0.5315, -0.1500, -0.3391, -0.8738],
         [ 0.4340, -0.4771,  0.2328,  0.3190, -0.2412, -0.1649, -0.7301],
         [ 0.4747, -0.3050, -0.0122,  0.3711, -0.0753, -0.2222, -0.8254],
         [ 0.4725, -0.4890,  0.0295,  0.2596, -0.1191, -0.1491, -0.9787],
         [ 0.5074, -0.1094, -0.0836,  0.3541, -0.0546, -0.1934, -0.6000],
         [ 0.3267, -0.5381,  0.3480,  0.7511, -0.1718, -0.5084, -0.9872],
         [ 0.4346, -0.1361,  0.0234,  0.6348, -0.0332, -0.4098, -0.7033],
         [ 0.3250, -0.7289,  0.5432,  0.5267, -0.3577, -0.3204, -0.8626],
         [ 0.4272, -0.5382,  0.2273,  0.3262, -0.2279, -0.1797, -0.8393],
         [ 0.4308, -0.2018,  0.1035,  0.5649, -0.1037, -0.3492, -0.6485],
         [ 0.4112, -0.5541,  0.3870,  

In [86]:
tests.test_linear_parameters(Linear, bias=False)
tests.test_linear_parameters(Linear, bias=True)
tests.test_linear_forward(Linear, bias=False)
tests.test_linear_forward(Linear, bias=True)

All tests in `test_linear_parameters` passed!
All tests in `test_linear_parameters` passed!
All tests in `test_linear_forward` passed!
All tests in `test_linear_forward` passed!


/tmp/ipykernel_526/3773404872.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


In [44]:
torch.tensor(torch.ones(5, 3)) - 0.5

/tmp/ipykernel_526/3058297645.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(torch.ones(5, 3)) - 0.5


tensor([[0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000]])

## Flatten

In [96]:
class Flatten(nn.Module):
    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: Tensor) -> Tensor:
        """
        Flatten out dimensions from start_dim to end_dim, inclusive of both.
        """
        shape = input.shape

        # Get start & end dims, handling negative indexing for end dim
        start_dim = self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim

        # Get the shapes to the left / right of flattened dims, as well as size of flattened middle
        shape_left = shape[:start_dim]
        shape_right = shape[end_dim + 1 :]
        shape_middle = t.prod(t.tensor(shape[start_dim : end_dim + 1])).item()

        return t.reshape(input, shape_left + (shape_middle,) + shape_right)

    def extra_repr(self) -> str:
        return ", ".join([f"{key}={getattr(self, key)}" for key in ["start_dim", "end_dim"]])


MNIST MLP

In [105]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = Flatten(-2, -1)
        self.linear1 = nn.Linear(28 * 28, 100)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(100, 10)

    def forward(self, x: Tensor) -> Tensor:
        ret = self.flatten(x)
        ret = self.linear1(ret)
        ret = self.relu(ret)
        ret = self.linear2(ret)

        return ret

    def __call__(self, x: Tensor) -> Tensor:
      return self.forward(x)

In [106]:
# Flatten()(torch.rand(28, 28), (28 * 28)).shape

In [117]:
mymlp = SimpleMLP()
mymlp(torch.rand(5, 28, 28))

imgs = torch.rand(5, 1, 28, 28)

mymlp(torch.rand(5, 1, 28, 28))

display(transforms.ToPILImage()(imgs[0]))

In [119]:
%cd /content/ARENA_materials/chapter0_fundamentals/exercises/part2_cnns

/content/ARENA_materials/chapter0_fundamentals/exercises/part2_cnns


In [120]:
tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

All tests in `test_mlp_module` passed!
All tests in `test_mlp_forward` passed!
